In [1]:
!cp -r /kaggle/input/datasets/hiemalrana22/quantum-spiking-nids-code /kaggle/working/qsnn_nids
!chmod -R u+w /kaggle/working/qsnn_nids

In [2]:
!ls /kaggle/working/qsnn_nids

artifacts  configs  data  qsnn	README.md  requirements.txt  scripts


In [3]:
!cd /kaggle/working/qsnn_nids && pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 85.9 MB/s eta 0:00:00


In [4]:
!python -c "import torch, pennylane; print('PyTorch:', torch.__version__); print('PennyLane:', pennylane.__version__)"

PyTorch: 2.10.0+cpu
PennyLane: 0.45.1


In [5]:
!cd /kaggle/working/qsnn_nids && python scripts/smoke_test.py


QSNN-NIDS smoke test (synthetic data, no download)

>>> config loads + n_qubits syncs with n_features
  [PASS] config loads + n_qubits syncs with n_features

>>> preprocessing produces [0,1] features of the right width
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (174). n_quantiles is set to n_samples.
  warnings.warn(
    train (174, 4)  val (60, 4)  test (120, 4)
  [PASS] preprocessing produces [0,1] features of the right width

>>> every spike encoder emits binary [B,T,F] tensors
    rate      shape (16, 4, 4)  sparsity 0.461
    latency   shape (16, 4, 4)  sparsity 0.184
    phase     shape (16, 4, 4)  sparsity 0.430
    delta     shape (16, 4, 4)  sparsity 0.668
    repeat    shape (16, 4, 4)  sparsity 0.449
  [PASS] every spike encoder emits binary [B,T,F] tensors

>>> flow windowing (sequence mode for CIC-IDS2017)
    windows (58, 5, 4)  attack fraction 0.638
  [PASS] flo

In [6]:
!find /kaggle/input -type f \( -iname "*training-set*.csv" -o -iname "*testing-set*.csv" \) | sort

/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_testing-set.csv
/kaggle/input/datasets/mrwellsdavid/unsw-nb15/UNSW_NB15_training-set.csv


In [7]:
!ls -lh /kaggle/input/datasets/mrwellsdavid/unsw-nb15

total 605M
-rw-r--r-- 1 nobody nogroup 4.0K Aug 11 11:23 NUSW-NB15_features.csv
-rw-r--r-- 1 nobody nogroup 162M Aug 11 11:24 UNSW-NB15_1.csv
-rw-r--r-- 1 nobody nogroup 158M Aug 11 11:23 UNSW-NB15_2.csv
-rw-r--r-- 1 nobody nogroup 148M Aug 11 11:23 UNSW-NB15_3.csv
-rw-r--r-- 1 nobody nogroup  94M Aug 11 11:23 UNSW-NB15_4.csv
-rw-r--r-- 1 nobody nogroup 4.6K Aug 11 11:23 UNSW-NB15_LIST_EVENTS.csv
-rw-r--r-- 1 nobody nogroup  31M Aug 11 11:23 UNSW_NB15_testing-set.csv
-rw-r--r-- 1 nobody nogroup  15M Aug 11 11:23 UNSW_NB15_training-set.csv


In [8]:
!cd /kaggle/working/qsnn_nids && python scripts/prepare_data.py --config configs/unsw_nb15.yaml --root /kaggle/input/datasets/mrwellsdavid/unsw-nb15 --max-rows 50000

[data]  UNSW_NB15_training-set.csv                          82,332 rows
[data]  combined: 50,000 rows x 45 cols
[prep]  after encoding: 36,419 x 75 (2 classes)
[prep]  MI filter -> 32 features; top-5: ['sbytes', 'smean', 'dbytes', 'ct_state_ttl', 'sload']
[prep]  PCA -> 8 dims (explained variance 0.915)
[prep]  bundle ready
  dataset      : UNSW-NB15
  features     : 8  ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8']
  classes      : 2  ['benign', 'attack']
  train/val/test: 21,218 / 3,642 / 7,284
  train balance: [10609, 10609]
  class weights: [1.0, 1.0]
[prep]  saved -> data/processed/unsw_nb15.npz


In [9]:
!ls -lh /kaggle/working/qsnn_nids/data/processed/

total 896K
-rw-r--r-- 1 root root 893K Sep  7 17:17 unsw_nb15.npz


In [10]:
!cd /kaggle/working/qsnn_nids && python scripts/train.py --config configs/unsw_nb15.yaml --bundle data/processed/unsw_nb15.npz --epochs 30 --device cpu --baselines


QSNN-NIDS  |  unsw_nb15  |  run 'qsnn_unsw'

------------------------------------------------------------------------------
MODEL
------------------------------------------------------------------------------
QSNN-NIDS
  input features        : 8
  spike encoding        : SpikeEncoder(scheme=phase, T=12, gain=1.0)
  qubits                : 8  (2 QLIF layers)
  circuit depth / step  : 15  (2q gates: 32)
  circuit evals / sample: 24
  params (classical/q)  : 244 / 96
  readout               : rate_mem_prob -> 2 classes
  noisy device          : False   diff: backprop

------------------------------------------------------------------------------
TRAINING
------------------------------------------------------------------------------
    ep  1 step   50  loss 0.1684
  epoch   1/30  loss 0.1468  val accuracy=0.8163  f1=0.7972  fpr=0.2201  roc_auc=0.9032  (233.7s)
    ep  2 step   50  loss 0.1071
  epoch   2/30  loss 0.1064  val accuracy=0.8215  f1=0.8042  fpr=0.2206  roc_auc=0.9090  (232.1

In [11]:
!find /kaggle/working/qsnn_nids/artifacts -type f | sort

/kaggle/working/qsnn_nids/artifacts/.gitkeep
/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/best.pt
/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/config.used.yaml
/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/final.pt
/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/history.json
/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/results.json
/kaggle/working/qsnn_nids/artifacts/smoke/history.json


In [12]:
!cat /kaggle/working/qsnn_nids/artifacts/qsnn_unsw/results.json

{
  "config": {
    "data": {
      "dataset": "unsw_nb15",
      "root": "data/raw/unsw_nb15",
      "cache_dir": "data/processed",
      "max_rows": 200000,
      "balance": "undersample",
      "binary": true,
      "min_class_count": 200,
      "scaler": "quantile",
      "select_k": 32,
      "reducer": "pca",
      "n_features": 8,
      "test_size": 0.2,
      "val_size": 0.1,
      "seed": 42,
      "n_jobs": -1
    },
    "encoding": {
      "scheme": "phase",
      "timesteps": 12,
      "gain": 1.0,
      "tau": 3.0,
      "deterministic": false,
      "seed": 42
    },
    "model": {
      "n_qubits": 8,
      "n_qlif_layers": 2,
      "ansatz_layers": 2,
      "ansatz": "strong",
      "beta": 0.9,
      "learn_beta": true,
      "threshold": 0.55,
      "learn_threshold": false,
      "reset": "subtract",
      "angle_map": "sigmoid",
      "angle_gain": 2.0,
      "angle_bias": -2.0,
      "surrogate": "fast_sigmoid",
      "surrogate_alpha": 5.0,
      "readout": "rate_

In [13]:
import json

path = "/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/results.json"

with open(path) as f:
    results = json.load(f)

print(json.dumps(results, indent=2))

{
  "config": {
    "data": {
      "dataset": "unsw_nb15",
      "root": "data/raw/unsw_nb15",
      "cache_dir": "data/processed",
      "max_rows": 200000,
      "balance": "undersample",
      "binary": true,
      "min_class_count": 200,
      "scaler": "quantile",
      "select_k": 32,
      "reducer": "pca",
      "n_features": 8,
      "test_size": 0.2,
      "val_size": 0.1,
      "seed": 42,
      "n_jobs": -1
    },
    "encoding": {
      "scheme": "phase",
      "timesteps": 12,
      "gain": 1.0,
      "tau": 3.0,
      "deterministic": false,
      "seed": 42
    },
    "model": {
      "n_qubits": 8,
      "n_qlif_layers": 2,
      "ansatz_layers": 2,
      "ansatz": "strong",
      "beta": 0.9,
      "learn_beta": true,
      "threshold": 0.55,
      "learn_threshold": false,
      "reset": "subtract",
      "angle_map": "sigmoid",
      "angle_gain": 2.0,
      "angle_bias": -2.0,
      "surrogate": "fast_sigmoid",
      "surrogate_alpha": 5.0,
      "readout": "rate_

In [14]:
!cd /kaggle/working/qsnn_nids && zip -r /kaggle/working/qsnn_results.zip artifacts/

  adding: artifacts/ (stored 0%)
  adding: artifacts/qsnn_unsw/ (stored 0%)
  adding: artifacts/qsnn_unsw/best.pt (deflated 50%)
  adding: artifacts/qsnn_unsw/history.json (deflated 74%)
  adding: artifacts/qsnn_unsw/final.pt (deflated 51%)
  adding: artifacts/qsnn_unsw/results.json (deflated 65%)
  adding: artifacts/qsnn_unsw/config.used.yaml (deflated 50%)
  adding: artifacts/smoke/ (stored 0%)
  adding: artifacts/smoke/history.json (deflated 67%)
  adding: artifacts/.gitkeep (stored 0%)


In [15]:
!ls -lh /kaggle/working/qsnn_results.zip

-rw-r--r-- 1 root root 18K Sep  7 18:47 /kaggle/working/qsnn_results.zip


In [16]:
!cd /kaggle/working/qsnn_nids && zip -r /kaggle/working/qsnn_processed_data.zip data/processed/

  adding: data/processed/ (stored 0%)
  adding: data/processed/unsw_nb15.npz (deflated 0%)
  adding: data/processed/.gitkeep (stored 0%)


In [17]:
!ls -lh /kaggle/working/*.zip

-rw-r--r-- 1 root root 893K Sep  7 18:47 /kaggle/working/qsnn_processed_data.zip
-rw-r--r-- 1 root root  18K Sep  7 18:47 /kaggle/working/qsnn_results.zip


In [18]:
!ls -lh /kaggle/working/qsnn_nids/artifacts/qsnn_unsw/

total 52K
-rw-r--r-- 1 root root 8.5K Sep  7 18:15 best.pt
-rw-r--r-- 1 root root 1.4K Sep  7 18:47 config.used.yaml
-rw-r--r-- 1 root root 8.7K Sep  7 18:47 final.pt
-rw-r--r-- 1 root root  16K Sep  7 18:46 history.json
-rw-r--r-- 1 root root 4.9K Sep  7 18:47 results.json


In [19]:
!cat /kaggle/working/qsnn_nids/artifacts/qsnn_unsw/config.used.yaml

data:
  dataset: unsw_nb15
  root: data/raw/unsw_nb15
  cache_dir: data/processed
  max_rows: 200000
  balance: undersample
  binary: true
  min_class_count: 200
  scaler: quantile
  select_k: 32
  reducer: pca
  n_features: 8
  test_size: 0.2
  val_size: 0.1
  seed: 42
  n_jobs: -1
encoding:
  scheme: phase
  timesteps: 12
  gain: 1.0
  tau: 3.0
  deterministic: false
  seed: 42
model:
  n_qubits: 8
  n_qlif_layers: 2
  ansatz_layers: 2
  ansatz: strong
  beta: 0.9
  learn_beta: true
  threshold: 0.55
  learn_threshold: false
  reset: subtract
  angle_map: sigmoid
  angle_gain: 2.0
  angle_bias: -2.0
  surrogate: fast_sigmoid
  surrogate_alpha: 5.0
  readout: rate_mem_prob
  dropout: 0.1
  n_classes: 2
  diff_method: backprop
noise:
  enabled: false
  backend: analytic
  depolarizing: 0.01
  amplitude_damping: 0.005
  phase_damping: 0.0
  readout_error: 0.0
  shots: null
  ibm_backend: ibm_brisbane
train:
  epochs: 30
  batch_size: 256
  lr: 0.003
  quantum_lr: 0.01
  weight_decay: 0.

In [20]:
import json

with open("/kaggle/working/qsnn_nids/artifacts/qsnn_unsw/history.json") as f:
    history = json.load(f)

print(json.dumps(history, indent=2))

[
  {
    "epoch": 1,
    "train_loss": 0.14679098655146225,
    "secs": 233.73142925399998,
    "val_accuracy": 0.8163097199341022,
    "val_balanced_accuracy": 0.8236412726464901,
    "val_precision": 0.7375210319685923,
    "val_recall": 0.8674142480211082,
    "val_f1": 0.7972112761442861,
    "val_f1_macro": 0.814665863919069,
    "val_mcc": 0.6382778304899138,
    "val_fpr": 0.22013170272812793,
    "val_fnr": 0.13258575197889183,
    "val_tnr_specificity": 0.779868297271872,
    "val_detection_rate": 0.8674142480211082,
    "val_roc_auc": 0.903241094676539,
    "val_pr_auc": 0.8478065948926998,
    "val_loss": 0.10344643096701907,
    "spike_rate": 0.295707605779171
  },
  {
    "epoch": 2,
    "train_loss": 0.10637017110521893,
    "secs": 232.10302470700003,
    "val_accuracy": 0.8215266337177375,
    "val_balanced_accuracy": 0.8300023952720061,
    "val_precision": 0.7400221729490022,
    "val_recall": 0.8806068601583114,
    "val_f1": 0.8042168674698795,
    "val_f1_macro": 

In [21]:
!cd /kaggle/working/qsnn_nids && python scripts/train.py --config configs/unsw_nb15.yaml --bundle data/processed/unsw_nb15.npz --epochs 30 --device cpu --baselines rf svm mlp


QSNN-NIDS  |  unsw_nb15  |  run 'qsnn_unsw'

------------------------------------------------------------------------------
MODEL
------------------------------------------------------------------------------
QSNN-NIDS
  input features        : 8
  spike encoding        : SpikeEncoder(scheme=phase, T=12, gain=1.0)
  qubits                : 8  (2 QLIF layers)
  circuit depth / step  : 15  (2q gates: 32)
  circuit evals / sample: 24
  params (classical/q)  : 244 / 96
  readout               : rate_mem_prob -> 2 classes
  noisy device          : False   diff: backprop

------------------------------------------------------------------------------
TRAINING
------------------------------------------------------------------------------
    ep  1 step   50  loss 0.1684
  epoch   1/30  loss 0.1468  val accuracy=0.8163  f1=0.7972  fpr=0.2201  roc_auc=0.9032  (230.5s)
    ep  2 step   50  loss 0.1071
  epoch   2/30  loss 0.1064  val accuracy=0.8215  f1=0.8042  fpr=0.2206  roc_auc=0.9090  (229.9